# How to build a PDF parser from scratch

---

## What We're Building Today

```
┌─────────────────────────┐          ┌─────────────────────────┐
│   PDF Document          │          │      Markdown File      │
│                         │   ───►   │                         │
│  Complex layouts,       │          │  # Clean headings       │
│  tables, fonts...       │          │  | Nice | Tables |      │
└─────────────────────────┘          └─────────────────────────┘
```

Notebook ini mencakup dua pendekatan utama dalam konversi PDF ke Markdown:
1. **Traditional Rule-Based Parsing** (PyMuPDF & pdfplumber) — Cepat, hemat biaya, dan presisi tinggi untuk dokumen digital.
2. **AI Vision Language Model (VLM) & Agentic Parsing** (Gemini Vision AI) — Memproses halaman secara visual untuk dokumen kompleks & scanned.


## The Roadmap

| Section | What We'll Do | Approach |
|---------|---------------|----------|
| **1. Setup & Environment** | Install & import libraries | Environment Setup |
| **2. Demystifying PDF Order** | Demo kegagalan `pypdf` pada reading order | Traditional |
| **3. Data Model** | Structure `TextBlock` & `TableBlock` | Traditional |
| **4. Digital Text Extraction** | Extract text & spatial coordinates via PyMuPDF | Traditional |
| **5. Spatial Reading Order** | Sort top-to-bottom, left-to-right | Traditional |
| **6. Heuristic Headings** | Font size analysis & heading classifier | Traditional |
| **7. Table Extraction** | Extract tables via `pdfplumber` | Traditional |
| **8. Pipeline Assembly** | Complete `pdf_to_markdown_smart()` pipeline | Traditional |
| **9. AI VLM Parsing** | Gemini Vision AI page-by-page parsing | AI VLM |
| **10. Agentic VLM Workflow** | Multi-agent layout analysis & validation | Agentic AI |
| **11. Comparison & Conclusion** | Benchmark Traditional vs VLM vs Agentic | Evaluation |


---

# 1. Setup & Environment

Kita menginstall library dasar untuk parsing tradisional (`pymupdf`, `pdfplumber`, `pypdf`) serta SDK Gemini AI (`google-generativeai`).


In [1]:
# Install required packages
!pip install pymupdf pdfplumber pypdf google-generativeai pillow -q


zsh:1: command not found: pip


In [2]:
import os
import re
import json
from dataclasses import dataclass
from typing import List, Optional, Tuple
from collections import defaultdict
import numpy as np
from PIL import Image

import fitz  # PyMuPDF
import pdfplumber
from pypdf import PdfReader
import google.generativeai as genai
from IPython.display import Markdown, display

print("✅ Semua library berhasil dimuat!")


✅ Semua library berhasil dimuat!


/var/folders/93/_s99tty12x14l_jrcg0zrmzw0000gn/T/ipykernel_14894/787799168.py:13: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


---

# 2. Demystifying PDF Order (Creation Order vs Reading Order)

PDF tidak menyimpan teks berdasarkan paragraf atau alur membaca, melainkan sebagai instruksi melukis elemen pada koordinat canvas. Library sederhana seperti `pypdf` membaca teks sesuai *creation order* internal, yang berantakan pada dokumen berkolom atau bertabel.


In [3]:
PDF_TEST = "readingOrder.pdf"

if os.path.exists(PDF_TEST):
    reader = PdfReader(PDF_TEST)
    page = reader.pages[0]
    raw_text = page.extract_text()

    print("❌ Output pypdf (Ekstraksi Naive Tanpa Koordinat):")
    print("=" * 65)
    print(raw_text[:800])
    print("=" * 65)
    print("⚠️ Teks berantakan karena pypdf mengikuti urutan internal PDF!")
else:
    print(f"File {PDF_TEST} tidak ditemukan.")


❌ Output pypdf (Ekstraksi Naive Tanpa Koordinat):
Users can easily manage email themself through 
web browser by web based interface.
Web mail
Hardware Specifications
Pentium III & Celeron up to 933MHz 
Intel 810e
8MB Flash ROM with embedded Linux OS
128MB SDRAM (PC133) up to 512MB
Standard 20GB HDD
(option RAID-1 mirroring with second HDD)
Dual 10/100 Base-T Ethernet
LCD Panel
9-pin serial port, 25-pin parallel port
DSL, Cable modem,Dial-up(internal 56k v.90)
12.0x3.5x15.75 inches
100-240V,50/60Hz
150 W
13.0 lbs(with one HDD)
Power, HDD Activity
FCC Class B, CE, TUV
FCC Class B, CE, TUV
Processor:
Chipset:
Disk-On-Chip:
Memory:
Storage:
Network Interface:
Display:
I/O Ports:
Connections:
Dimensions:
Power Requirements:
Power Consumption:
LEDs:
EMI:
Safety:
Weight: 
Web-based interface
Linux (kernel 2.4.14)
UI
OS
Service Web 
Sever DNS DH
⚠️ Teks berantakan karena pypdf mengikuti urutan internal PDF!


---

# 3. Data Model (`TextBlock` & `TableBlock`)

Untuk menyusun kembali urutan teks secara presisi, kita menyimpan koordinat `(x0, y0, x1, y1)`, ukuran font, dan atribut gaya pada kelas data berikut:


In [4]:
@dataclass
class TextBlock:
    text: str
    x0: float
    y0: float
    x1: float
    y1: float
    font_size: float
    font_name: str
    is_bold: bool
    page_num: int
    
    @property
    def center_y(self) -> float:
        return (self.y0 + self.y1) / 2
    
    @property
    def center_x(self) -> float:
        return (self.x0 + self.x1) / 2


@dataclass
class TableBlock:
    data: List[List[str]]
    x0: float
    y0: float
    x1: float
    y1: float
    page_num: int
    
    @property
    def center_y(self) -> float:
        return (self.y0 + self.y1) / 2

print("✅ Data Model TextBlock & TableBlock siap!")


✅ Data Model TextBlock & TableBlock siap!


---

# 4. Digital Text Extraction Layer (PyMuPDF)

Fungsi `extract_text_blocks()` membaca PDF digital menggunakan PyMuPDF dan mengambil setiap span teks beserta koordinat dan font.


In [5]:
def clean_ocr_typos(text: str) -> str:
    """Pembersihan awal karakter noise."""
    if not text:
        return ""
    return text.strip()


def extract_text_blocks(pdf_path: str) -> List[TextBlock]:
    """Ekstraksi blok teks digital dengan koordinat spatial."""
    doc = fitz.open(pdf_path)
    text_blocks = []
    
    for page_num, page in enumerate(doc):
        blocks = page.get_text("dict")["blocks"]
        for block in blocks:
            if block["type"] == 0:  # Text block
                for line in block["lines"]:
                    for span in line["spans"]:
                        text = clean_ocr_typos(span["text"])
                        if not text:
                            continue
                        
                        font_name = span["font"].lower()
                        is_bold = "bold" in font_name or bool(span["flags"] & 2**4)
                        
                        text_blocks.append(TextBlock(
                            text=text,
                            x0=span["bbox"][0],
                            y0=span["bbox"][1],
                            x1=span["bbox"][2],
                            y1=span["bbox"][3],
                            font_size=span["size"],
                            font_name=span["font"],
                            is_bold=is_bold,
                            page_num=page_num
                        ))
    doc.close()
    return text_blocks

# 🧪 SUBTEST: Verify PyMuPDF Raw Extraction
if os.path.exists("readingOrder.pdf"):
    sample_b = extract_text_blocks("readingOrder.pdf")
    print(f"Berhasil mengekstrak {len(sample_b)} blok teks dari readingOrder.pdf")
    for b in sample_b[:3]:
        print(f"  Page {b.page_num} | Y={b.y0:.1f}, X={b.x0:.1f} | Font={b.font_size:.1f}pt | Text: '{b.text[:30]}...'")


Berhasil mengekstrak 145 blok teks dari readingOrder.pdf
  Page 0 | Y=509.6, X=218.7 | Font=7.5pt | Text: 'Users can easily manage email ...'
  Page 0 | Y=518.6, X=218.7 | Font=7.5pt | Text: 'web browser by web based inter...'
  Page 0 | Y=499.5, X=218.7 | Font=7.5pt | Text: 'Web mail...'


---

# 5. Spatial Reading Order & Line Grouping

Mengelompokkan kata yang berada dalam toleransi Y sama menjadi satu baris horizontal, kemudian mengurutkannya dari atas ke bawah (Y0) dan kiri ke kanan (X0).


In [6]:
def group_blocks_into_lines(blocks: List[TextBlock], tolerance: float = 6.0) -> List[TextBlock]:
    """Mengelompokkan kata-kata pada baris horizontal yang sama."""
    if not blocks:
        return []
        
    sorted_blocks = sorted(blocks, key=lambda b: (b.page_num, b.y0, b.x0))
    lines = []
    current_line = [sorted_blocks[0]]
    
    for block in sorted_blocks[1:]:
        last = current_line[-1]
        if block.page_num == last.page_num and abs(block.center_y - last.center_y) <= tolerance:
            current_line.append(block)
        else:
            current_line.sort(key=lambda b: b.x0)
            merged_text = " ".join(b.text for b in current_line)
            lines.append(TextBlock(
                text=merged_text,
                x0=current_line[0].x0,
                y0=min(b.y0 for b in current_line),
                x1=current_line[-1].x1,
                y1=max(b.y1 for b in current_line),
                font_size=max(b.font_size for b in current_line),
                font_name=current_line[0].font_name,
                is_bold=any(b.is_bold for b in current_line),
                page_num=current_line[0].page_num
            ))
            current_line = [block]
            
    if current_line:
        current_line.sort(key=lambda b: b.x0)
        merged_text = " ".join(b.text for b in current_line)
        lines.append(TextBlock(
            text=merged_text,
            x0=current_line[0].x0,
            y0=min(b.y0 for b in current_line),
            x1=current_line[-1].x1,
            y1=max(b.y1 for b in current_line),
            font_size=max(b.font_size for b in current_line),
            font_name=current_line[0].font_name,
            is_bold=any(b.is_bold for b in current_line),
            page_num=current_line[0].page_num
        ))
        
    return lines


def sort_blocks_reading_order(blocks: List[TextBlock]) -> List[TextBlock]:
    """Pengurutan teks sesuai urutan baca alami."""
    grouped = group_blocks_into_lines(blocks, tolerance=6.0)
    return sorted(grouped, key=lambda b: (b.page_num, b.y0, b.x0))

print("✅ Reading Order Sorting Function Siap!")


✅ Reading Order Sorting Function Siap!


---

# 6. Heuristic Heading Classification & Font Analysis

Analisis distribusi statistik ukuran font untuk menentukan `body_size` secara dinamis dan memetakan ukuran font yang lebih besar ke level heading `# H1` hingga `### H3`.


In [7]:
def analyze_font_sizes(blocks: List[TextBlock]) -> dict:
    """Menghitung distribusi ukuran font pada dokumen."""
    if not blocks:
        return {"body_size": 12, "heading_sizes": []}
        
    size_counts = defaultdict(int)
    for block in blocks:
        rounded = round(block.font_size, 1)
        size_counts[rounded] += len(block.text)
        
    body_size = max(size_counts.items(), key=lambda x: x[1])[0]
    heading_sizes = sorted([s for s in size_counts.keys() if s > body_size * 1.25], reverse=True)
    
    return {"body_size": body_size, "heading_sizes": heading_sizes}


def classify_heading_level(block: TextBlock, font_analysis: dict) -> Optional[int]:
    """Klasifikasi level heading berdasarkan rasio ukuran font & panjang teks."""
    text = clean_ocr_typos(block.text)
    if not text:
        return None
        
    body_size = font_analysis.get("body_size", 12)
    size_ratio = block.font_size / body_size if body_size > 0 else 1.0
    text_len = len(text)
    
    section_pattern = re.match(r'^(Pasal\s+\d+|BAB\s+[IVX0-9]+|Chapter\s+\d+|(\d+\.)+\d*\s+)', text, re.IGNORECASE)
    if section_pattern and text_len <= 80:
        if 'pasal' in text.lower() or 'bab' in text.lower() or size_ratio >= 1.3:
            return 1
        return 2

    if text_len > 75:
        return None
        
    if size_ratio >= 1.5 and text_len <= 50:
        return 1
    elif size_ratio >= 1.25 and text_len <= 65:
        return 2
    elif block.is_bold and size_ratio >= 1.1 and text_len <= 70:
        return 3
        
    return None


def text_to_markdown(block: TextBlock, heading_level: Optional[int]) -> str:
    """Konversi blok teks ke format Markdown."""
    text = clean_ocr_typos(block.text)
    if heading_level:
        prefix = "#" * heading_level
        return f"{prefix} {text}"
    else:
        return text

print("✅ Font Size Analysis & Heading Classifier Siap!")


✅ Font Size Analysis & Heading Classifier Siap!


---

# 7. Table Extraction (`pdfplumber`) & Markdown Formatting

Menggunakan `pdfplumber` untuk mengekstrak struktur sel tabel dari PDF digital dan mengonversinya ke dalam tabel Markdown (`| ... |`).


In [8]:
def extract_tables(pdf_path: str) -> List[TableBlock]:
    """Ekstraksi tabel terstruktur menggunakan pdfplumber."""
    tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            page_tables = page.find_tables()
            for table in page_tables:
                data = table.extract()
                if data and len(data) > 0:
                    bbox = table.bbox
                    tables.append(TableBlock(
                        data=data,
                        x0=bbox[0],
                        y0=bbox[1],
                        x1=bbox[2],
                        y1=bbox[3],
                        page_num=page_num
                    ))
    return tables


def table_to_markdown(table: TableBlock) -> str:
    """Konversi objek TableBlock ke sintaks Markdown Table."""
    if not table.data or len(table.data) == 0:
        return ""
        
    max_cols = max(len(row) for row in table.data)
    if max_cols == 0:
        return ""
        
    lines = []
    for row_idx, row in enumerate(table.data):
        cells = []
        for cell in row:
            cell_text = str(cell) if cell is not None else ""
            cell_text = cell_text.replace("\n", " ").replace("|", "\|").strip()
            cells.append(cell_text)
            
        while len(cells) < max_cols:
            cells.append("")
            
        line = "| " + " | ".join(cells) + " |"
        lines.append(line)
        
        if row_idx == 0:
            separator = "| " + " | ".join(["---"] * max_cols) + " |"
            lines.append(separator)
            
    return "\n".join(lines)

print("✅ Table Extractor (pdfplumber) & Markdown Table Generator Siap!")


✅ Table Extractor (pdfplumber) & Markdown Table Generator Siap!


---

# 8. Duplicate Filtering & Paragraph Assembly

- `is_inside_table()`: Membuang teks yang sudah menjadi bagian dari tabel agar tidak terjadi duplikasi.
- `merge_consecutive_text()`: Menggabungkan potongan baris teks menjadi paragraf utuh.


In [9]:
def is_inside_table(block: TextBlock, tables: List[TableBlock]) -> bool:
    """Filter untuk mencegah teks di dalam tabel terekstrak ganda."""
    for table in tables:
        if (block.page_num == table.page_num and
            block.x0 >= table.x0 - 5 and block.x1 <= table.x1 + 5 and
            block.y0 >= table.y0 - 5 and block.y1 <= table.y1 + 5):
            return True
    return False


def merge_consecutive_text(blocks: List[TextBlock], font_analysis: dict) -> List[TextBlock]:
    """Menggabungkan baris-baris paragraf secara kontinu."""
    if not blocks:
        return []
        
    merged = []
    current_paragraph = []
    
    for block in blocks:
        level = classify_heading_level(block, font_analysis)
        if level:
            if current_paragraph:
                merged_text = " ".join(b.text for b in current_paragraph)
                merged.append(TextBlock(
                    text=merged_text,
                    x0=current_paragraph[0].x0,
                    y0=current_paragraph[0].y0,
                    x1=current_paragraph[-1].x1,
                    y1=current_paragraph[-1].y1,
                    font_size=current_paragraph[0].font_size,
                    font_name=current_paragraph[0].font_name,
                    is_bold=False,
                    page_num=current_paragraph[0].page_num
                ))
                current_paragraph = []
            merged.append(block)
        else:
            current_paragraph.append(block)
            
    if current_paragraph:
        merged_text = " ".join(b.text for b in current_paragraph)
        merged.append(TextBlock(
            text=merged_text,
            x0=current_paragraph[0].x0,
            y0=current_paragraph[0].y0,
            x1=current_paragraph[-1].x1,
            y1=current_paragraph[-1].y1,
            font_size=current_paragraph[0].font_size,
            font_name=current_paragraph[0].font_name,
            is_bold=False,
            page_num=current_paragraph[0].page_num
        ))
        
    return merged

print("✅ Filtering & Paragraph Merger Siap!")


✅ Filtering & Paragraph Merger Siap!


---

# 9. Complete Traditional Pipeline Assembly (`pdf_to_markdown_smart`)

Fungsi utama yang menggabungkan ekstraksi teks digital, ekstraksi tabel, pengurutan spatial, klasifikasi heading, dan perakitan Markdown.


In [10]:
def pdf_to_markdown_smart(pdf_path: str) -> str:
    """Konversi PDF Digital ke Markdown secara Cerdas & Terstruktur."""
    print(f"📄 Processing: {pdf_path}")
    
    text_blocks = extract_text_blocks(pdf_path)
    tables = extract_tables(pdf_path)
    
    font_analysis = analyze_font_sizes(text_blocks)
    
    if tables:
        text_blocks = [b for b in text_blocks if not is_inside_table(b, tables)]
        
    markdown_parts = []
    all_pages = sorted(list(set(b.page_num for b in text_blocks) | set(t.page_num for t in tables)))
    
    for idx, page_num in enumerate(all_pages):
        if idx > 0:
            markdown_parts.append(f"\n\n---\n\n> **📄 Halaman {page_num + 1}**\n\n")
            
        page_text = [b for b in text_blocks if b.page_num == page_num]
        page_tables = [t for t in tables if t.page_num == page_num]
        
        page_text = sort_blocks_reading_order(page_text)
        page_text = merge_consecutive_text(page_text, font_analysis)
        
        elements = []
        for block in page_text:
            level = classify_heading_level(block, font_analysis)
            md = text_to_markdown(block, level)
            elements.append((block.y0, 'text', md))
            
        for table in page_tables:
            md = table_to_markdown(table)
            if md.strip():
                elements.append((table.y0, 'table', md))
                
        elements.sort(key=lambda x: x[0])
        
        for _, elem_type, content in elements:
            if elem_type == 'table':
                markdown_parts.append("")
                markdown_parts.append(content)
                markdown_parts.append("")
            else:
                markdown_parts.append(content)
                
    return "\n\n".join(markdown_parts)

print("✅ Pipeline pdf_to_markdown_smart() SIAP!")


✅ Pipeline pdf_to_markdown_smart() SIAP!


In [11]:
# 🧪 LIVE DEMO: Run Traditional Parser on Sample Documents
TEST_DOCS = ["readingOrder.pdf", "colSample.pdf", "nvidia10q.pdf"]

for pdf in TEST_DOCS:
    if os.path.exists(pdf):
        print(f"\n{'='*60}\nTesting: {pdf}\n{'='*60}")
        output_md = pdf_to_markdown_smart(pdf)
        display(Markdown(output_md[:1500] + "\n\n*(output dipotong untuk preview)...*"))
    else:
        print(f"File {pdf} tidak ditemukan.")



Testing: readingOrder.pdf
📄 Processing: readingOrder.pdf




| H U M M I N G |  |  |  |
| --- | --- | --- | --- |
|  |  |  |  |



## Software Features

Auto-Configuration Network Addess Translation (NAT) Hardware Specifications DHCP configures your network Use a single IP address for the entire automatically. network and mask clients on the Rescue System network from the Internet. Disk-On-Chip(DOC) can reboot the system in the unlikely event of a Cross-Platform Support system crash. Exchange information effortlessly Email Service Scheduled mail retrieal and manage- between Windows,Unix and Linux clients. ment witch support for SMTP and Multilingual User Interface POP3 clients. Switch menus between English, Web server Simplified Chinese,Traditional Web hosting and modification with Chinese fonts. Online Software Upgrade support for MS Frontpage extensions. Cache frequently accessed pages to reduce traffic and enhance network performance. RAID-1 Mirroring Protect critical data in the event of hard drive failure through redundancy over two drives. Processor: Pentium III & Celeron up to 933MHz Chipset: Intel 810e Disk-On-Chip: 8MB Flash ROM with embedded Linux OS Remotely upgrade software services via Internet to keep up with leading technology. DNS Easy management of your domain Memory: 128MB SDRAM (PC133) up to 512MB Storage: name settings Standard 20GB HDD Network Resource Sharing (option RAID-1 mirroring with second HDD) Share files on your network. Network Interface: Dual 10/100 Base-T Ethernet Display: LCD Panel I/O Ports: 9-pin s

*(output dipotong untuk preview)...*


Testing: colSample.pdf
📄 Processing: colSample.pdf


Hyung Won Chung, Le Hou, S. Longpre, Barret Jason Phang, Laria Reynolds, Eric Tang, Anish Thite, Zoph, Yi Tay, William Fedus, Eric Li, Xuezhi Ben Wang, Kevin Wang, and Andy Zou. 2021. A Wang, Mostafa Dehghani, Siddhartha Brahma, Al- framework for few-shot language model evaluation . bert Webson, Shixiang Shane Gu, Zhuyun Dai, Mirac Suzgun, Xinyun Chen, Aakanksha Chowd- Samuel Gehman, Suchin Gururangan, Maarten Sap, hery, Dasha Valter, Sharan Narang, Gaurav Mishra, Yejin Choi, and Noah A Smith. 2020. Realtoxici- Adams Wei Yu, Vincent Zhao, Yanping Huang, An- typrompts: Evaluating neural toxic degeneration in drew M. Dai, Hongkun Yu, Slav Petrov, Ed Huai language models. arXiv preprint arXiv:2009.11462 . hsin Chi, Jeff Dean, Jacob Devlin, Adam Roberts, Alex Graves. 2013. Generating sequences with Denny Zhou, Quoc Le, and Jason Wei. 2022. Scal- recurrent neural networks. arXiv preprint ing instruction-ﬁnetuned language models. arXiv arXiv:1308.0850 . preprint arXiv:2210.11416 . Kenneth Heaﬁeld, Ivan Pouzyrevsky, Jonathan H Clark, Christopher Clark, Kenton Lee, Ming-Wei Chang, and Philipp Koehn. 2013. Scalable modiﬁed kneser- Tom Kwiatkowski, Michael Collins, and Kristina ney language model estimation. In Proceedings Toutanova. 2019. Boolq: Exploring the surprising of the 51st Annual Meeting of the Association for difﬁculty of natural yes/no questions. arXiv preprint Computational Linguistics (Volume 2: Short Papers) , arXiv:1905.10044 . pages 690–696. Peter Clark, Isaac Cowhey, 

*(output dipotong untuk preview)...*


Testing: nvidia10q.pdf
📄 Processing: nvidia10q.pdf


UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549

## FORM 10-Q

☒ QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the quarterly period ended October 26, 2025 OR ☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 Commission File Number: 0-23985

## NVIDIA CORPORATION

(Exact name of registrant as specified in its charter) Delaware 94-3177549 (State or other jurisdiction of (I.R.S. Employer incorporation or organization) Identification No.) 2788 San Tomas Expressway, Santa Clara, California 95051 (Address of principal executive offices) (Zip Code) (408) 486-2000 (Registrant's telephone number, including area code) N/A (Former name, former address and former fiscal year, if changed since last report) Securities registered pursuant to Section 12(b) of the Act: Title of each class Trading Symbol(s) Name of each exchange on which registered Common Stock, $0.001 par value per share NVDA The Nasdaq Global Select Market Indicate by check mark whether the registrant (1) has filed all reports required to be filed by Section 13 or 15(d) of the Securities Exchange Act of 1934 during the preceding 12 months (or for such shorter period that the registrant was required to file such reports), and (2) has been subject to such filing requirements for the past 90 days. Yes ☒ No ☐ Indicate by check mark whether the registrant has submitted electronically every Interactive Data File required 

*(output dipotong untuk preview)...*

---

# 10. AI / Vision Language Model (VLM) Parsing Layer (Gemini Vision AI)

Untuk dokumen kompleks, bertata letak bebas, atau scanned PDF, kita dapat memanfaatkan **Vision Language Model (VLM)** seperti Gemini 1.5 Pro / Flash yang membaca visual halaman secara menyeluruh.


In [12]:
# 🔑 Setup Gemini API Key
api_key = os.getenv("GEMINI_API_KEY", "")

if api_key:
    genai.configure(api_key=api_key)
    print("✅ Gemini API Key terdeteksi dan berhasil dikonfigurasi!")
else:
    print("ℹ️ GEMINI_API_KEY tidak ditemukan pada environment variables.")
    print("👉 Untuk mengaktifkan VLM, silakan jalankan: os.environ['GEMINI_API_KEY'] = 'YOUR_KEY'")


ℹ️ GEMINI_API_KEY tidak ditemukan pada environment variables.
👉 Untuk mengaktifkan VLM, silakan jalankan: os.environ['GEMINI_API_KEY'] = 'YOUR_KEY'


In [13]:
def pdf_to_markdown_vlm_gemini(pdf_path: str, model_name: str = "gemini-1.5-flash") -> str:
    """Konversi PDF ke Markdown berbasis Visual AI Gemini."""
    if not os.path.exists(pdf_path):
        return f"File {pdf_path} tidak ditemukan."
        
    doc = fitz.open(pdf_path)
    model = genai.GenerativeModel(model_name)
    markdown_results = []
    
    prompt = """
    You are an expert document parser. Convert this PDF page image into clean, accurately formatted Markdown.
    Follow these instructions carefully:
    1. Reconstruct all headings (#, ##, ###) based on visual hierarchy.
    2. Extract all tables into clean Markdown tables (| col1 | col2 |).
    3. Retain exact list numbering and bullet points.
    4. Do NOT omit any text or content.
    5. Output ONLY the raw Markdown without wrapping in extra markdown codeblocks.
    """
    
    print(f"🤖 Memproses {pdf_path} dengan Gemini Vision ({len(doc)} halaman)...")
    
    for page_num in range(len(doc)):
        pix = doc[page_num].get_pixmap(dpi=200)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        
        response = model.generate_content([prompt, img])
        page_md = response.text if response and hasattr(response, 'text') else ""
        
        markdown_results.append(f"<!-- Page {page_num + 1} -->\n" + page_md)
        print(f"   - Halaman {page_num + 1} selesai diproses.")
        
    doc.close()
    return "\n\n---\n\n".join(markdown_results)

print("✅ Fungsi pdf_to_markdown_vlm_gemini() siap!")


✅ Fungsi pdf_to_markdown_vlm_gemini() siap!


In [14]:
# 🧪 LIVE DEMO: Run Gemini VLM Parser (Optional)
if os.getenv("GEMINI_API_KEY") and os.path.exists("readingOrder.pdf"):
    vlm_output = pdf_to_markdown_vlm_gemini("readingOrder.pdf")
    print("\n📄 Result Gemini Vision Parsing:")
    print("=" * 60)
    display(Markdown(vlm_output[:2000]))
else:
    print("ℹ️ Lewati demo Gemini VLM (Sediakan GEMINI_API_KEY untuk menjalankan VLM secara langsung).")


ℹ️ Lewati demo Gemini VLM (Sediakan GEMINI_API_KEY untuk menjalankan VLM secara langsung).


---

# 11. Agentic VLM Parser Workflow

Pendekatan **Agentic VLM** membagi proses parsing menjadi alur multi-agen:
1. **Layout Inspector Agent**: Menganalisis wilayah visual (Header, Footer, Table, Column).
2. **Specialized Extractor Agent**: Mengekstrak teks & tabel sesuai tipe wilayahnya.
3. **Markdown Synthesizer Agent**: Menggabungkan hasil ekstraksi dan memverifikasi kualitas.


In [15]:
class AgenticVLMParserWorkflow:
    """Alur Parser Berbasis Agen Multi-Tahap."""
    def __init__(self, api_key: str = ""):
        self.api_key = api_key or os.getenv("GEMINI_API_KEY", "")
        if self.api_key:
            genai.configure(api_key=self.api_key)
            self.model = genai.GenerativeModel("gemini-1.5-flash")
            
    def analyze_layout(self, image: Image.Image) -> str:
        """Agent 1: Analisis Struktur Visual Halaman."""
        if not self.api_key: return ""
        prompt = "Analyze the visual layout of this page. Identify headers, multi-column areas, and tables."
        res = self.model.generate_content([prompt, image])
        return res.text if hasattr(res, 'text') else ""

    def extract_content(self, image: Image.Image, layout_info: str) -> str:
        """Agent 2: Ekstraksi Teks Berdasarkan Panduan Layout."""
        if not self.api_key: return ""
        prompt = f"Using this layout analysis: {layout_info}\nExtract all content into accurate Markdown."
        res = self.model.generate_content([prompt, image])
        return res.text if hasattr(res, 'text') else ""

    def process_pdf(self, pdf_path: str) -> str:
        if not os.path.exists(pdf_path):
            return "File tidak ditemukan."
        doc = fitz.open(pdf_path)
        pages_md = []
        for page_num in range(len(doc)):
            pix = doc[page_num].get_pixmap(dpi=200)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            layout = self.analyze_layout(img)
            md = self.extract_content(img, layout)
            pages_md.append(md)
        doc.close()
        return "\n\n---\n\n".join(pages_md)

print("✅ AgenticVLMParserWorkflow siap!")


✅ AgenticVLMParserWorkflow siap!


---

# 12. Evaluation & Comparison Matrix

| Matriks Evaluasi | Traditional Rule-Based (PyMuPDF + pdfplumber) | AI VLM (Gemini Vision AI) | Agentic VLM Workflow |
|------------------|-----------------------------------------------|---------------------------|----------------------|
| **Kecepatan** | ⚡ Sangat Cepat (< 1 detik/halaman) | 🐢 Sedang (3-5 detik/halaman) | 🐢 Lambat (8-15 detik/halaman) |
| **Biaya API** | 🆓 Gratis (Lokal) | 💳 Berbayar Per Token | 💳 Berbayar Multi-Call |
| **Dokumen Scanned**| ❌ Tidak Bisa (Membutuhkan OCR) | ✅ Berhasil Sangat Baik | ✅ Berhasil Presisi Tinggi |
| **Tabel Kompleks** | ✅ Terstruktur Baik (`pdfplumber`) | ⚠️ Tergantung Prompt VLM | ✅ Sangat Akurat |
| **Keamanan Data** | 🔒 100% Pengolahan Lokal | ☁️ Mengirim Gambar ke Cloud | ☁️ Mengirim Gambar ke Cloud |

---

### 💡 Rekomendasi Penggunaan:
- Gunakan **Traditional Parser** untuk dokumen digital berkecepatan tinggi & pengolahan data lokal secara gratis.
- Gunakan **Gemini VLM / Agentic VLM** untuk dokumen scan fisik, formulir tangan, atau tata letak visual majalah/jurnal kompleks!
